In [ ]:
from parse import parse_lammps, parse_nist
from encode import encode_data
from RAG_TECHNIQUES.helper_functions import *
import json

with open("PotentialData/nist_potentials.json", "r") as f:
    all_potentials = json.load(f)

chunks_vector_store = encode_data(all_potentials)

chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

score = 0
responses = []
for element in ["Al", "C", "Be", "Si", "Na", "K", "Ca"]:
    with open(f"Questions/generated_questions_{element}.json", "r") as f:
        questions = json.load(f)
    
    for question in questions:
        context = retrieve_context_per_question(question['question'], chunks_query_retriever)
        answer_list = []
        for ctx in context:
            ctx = ast.literal_eval(ctx)
            answer_list.append(ctx['id'])

        parity = "Incorrect"
        if len(answer_list) > 0 and question['potential'] in answer_list:
            score += 1
            parity = "Correct"

        responses.append({
            "question": question['question'],
            "expected_answer": question['potential'],
            "retrieved_answers": answer_list,
            "parity": parity
        })

print(f"Total Score: {score} out of {len(responses)}")
with open("responses.json", "w") as f:
    json.dump(responses, f, indent=4)